# MELI Shipping Forecast — API Usage Example

This notebook demonstrates how to consume the shipping demand forecasting API.
It covers all three endpoints:

- `GET /v1/health` — liveness + readiness probe
- `GET /v1/model/info` — model metadata and evaluation metrics
- `POST /v1/predict` — demand forecasting with optional intervals and cost-aware recommendations

## Prerequisites

Before running this notebook, start the API server in a separate terminal:

```bash
make train-model          # generate artifacts/lightgbm_final.{joblib,json}
uv run uvicorn shipping_forecast.api.app:app --port 8000
```

The notebook assumes the server is running at `http://127.0.0.1:8000`.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
import requests

BASE_URL = "http://127.0.0.1:8000"
session = requests.Session()
session.headers.update({"Content-Type": "application/json"})

print("Libraries loaded. Make sure the API server is running at", BASE_URL)

## 1. Health Check

The `/v1/health` endpoint confirms the service is running and the model is loaded.

In [ ]:
resp = session.get(f"{BASE_URL}/v1/health")
resp.raise_for_status()
print(f"Status: {resp.status_code}")
print(resp.json())

## 2. Model Info

The `/v1/model/info` endpoint exposes the public model metadata: version, training dates, known states, and honest out-of-sample evaluation metrics.

In [ ]:
resp = session.get(f"{BASE_URL}/v1/model/info")
resp.raise_for_status()
info = resp.json()

print(f"Model version : {info['model_version']}")
print(f"Trained at    : {info['trained_at']}")
print(f"Last train date: {info['last_train_date']}")
print(f"Data cutoff   : {info['data_cutoff']}")
print(f"N groups      : {info['n_groups']} states")
print()
print("Evaluation metrics (strict no-leakage holdout):")
metrics = info["evaluation_metrics"]
print(f"  Window : {metrics['window_start']} to {metrics['window_end']} ({metrics['n_days']} days)")
print(f"  WAPE   : {metrics['wape']:.4f}")
print(f"  MAE    : {metrics['mae']:.3f}")
print(f"  RMSE   : {metrics['rmse']:.3f}")

## 3. Basic Prediction

A simple 7-day forecast for São Paulo (SP) and Rio de Janeiro (RJ), using the server's default cost parameters.

In [ ]:
payload = {"start_date": "2018-09-01", "end_date": "2018-09-07", "states": ["SP", "RJ"]}

resp = session.post(f"{BASE_URL}/v1/predict", json=payload)
resp.raise_for_status()
data = resp.json()

print(f"Model version : {data['model_version']}")
print(f"N predictions : {data['metadata']['n_predictions']}")
print(f"Alpha source  : {data['metadata']['alpha_source']}")
print()

df = pd.DataFrame(data["predictions"])
df["date"] = pd.to_datetime(df["date"])
df

## 4. Cost-Aware Prediction

The `alpha` parameter controls the asymmetric safety margin:

- `alpha > 0`: penalize **under-prediction** (shift `recommended` toward the upper bound). Use when running out of capacity is more costly than over-provisioning.
- `alpha < 0`: penalize **over-prediction** (shift toward the lower bound). Use when over-provisioning wastes significant resources.
- `alpha = 0`: `recommended == point` (no adjustment).

Here we compare three alpha values for SP over 30 days.

In [ ]:
frames = {}
for alpha in [0.0, 0.65, 1.0]:
    resp = session.post(
        f"{BASE_URL}/v1/predict",
        json={
            "start_date": "2018-09-01",
            "end_date": "2018-09-30",
            "states": ["SP"],
            "alpha": alpha,
        },
    )
    resp.raise_for_status()
    df_alpha = pd.DataFrame(resp.json()["predictions"])
    df_alpha["date"] = pd.to_datetime(df_alpha["date"])
    frames[alpha] = df_alpha

print("Predictions loaded for alpha in [0.0, 0.65, 1.0]")

## 5. Visualization — 30-day forecast for SP with confidence band

The shaded area shows the 90% conformal prediction interval. The `recommended` line shifts toward the upper bound as `alpha` increases, reflecting higher under-prediction penalty.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
colors = {0.0: "#2196F3", 0.65: "#FF9800", 1.0: "#F44336"}
labels = {
    0.0: "alpha=0.0 (no adjustment)",
    0.65: "alpha=0.65 (default, moderate under-prediction penalty)",
    1.0: "alpha=1.0 (max penalty, recommended=upper bound)",
}

for ax, (alpha, df_a) in zip(axes, frames.items(), strict=False):
    color = colors[alpha]
    ax.fill_between(
        df_a["date"],
        df_a["lower_90"],
        df_a["upper_90"],
        alpha=0.2,
        color=color,
        label="90% interval",
    )
    ax.plot(
        df_a["date"],
        df_a["point"],
        color="#555555",
        linewidth=1.5,
        linestyle="--",
        label="point forecast",
    )
    ax.plot(df_a["date"], df_a["recommended"], color=color, linewidth=2, label="recommended")
    ax.set_title(labels[alpha], fontsize=10)
    ax.set_ylabel("Shipments")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle(
    "SP — 30-day shipping demand forecast\n(Conformal intervals, alpha comparison)", fontsize=12
)
plt.tight_layout()
plt.savefig("sp_forecast_alpha_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to sp_forecast_alpha_comparison.png")

## 6. All-States Snapshot

A single-day forecast for all 27 Brazilian states, sorted by recommended volume.

In [ ]:
resp = session.post(
    f"{BASE_URL}/v1/predict",
    json={"start_date": "2018-09-01", "end_date": "2018-09-01", "alpha": 0.65},
)
resp.raise_for_status()
df_all = pd.DataFrame(resp.json()["predictions"])
df_all = df_all.sort_values("recommended", ascending=False).reset_index(drop=True)
df_all[["state", "point", "lower_90", "upper_90", "recommended"]].round(1)